# Azure Retail Prices Fetcher

## Overview

This notebook provides a streamlined approach to download and analyze Azure pricing data using the official Azure Prices API. It enables you to programmatically access real-time retail pricing information for Azure services and resources without requiring authentication.

## Key Features

- **No Authentication Required**: Access public Azure pricing data freely
- **Real-time Data**: Retrieve the latest Azure service pricing directly from Microsoft
- **Flexible Filtering**: Query specific services, regions, and pricing tiers
- **Data Analysis Ready**: Clean, structured output optimized for analysis and visualization

## Resources

| Resource | Link |
|----------|------|
| **API Documentation** | [Azure Retail Prices REST API](https://learn.microsoft.com/en-us/rest/api/cost-management/retail-prices/azure-retail-prices) |
| **Base URL** | `https://prices.azure.com/api/retail/prices` |


In [1]:
import datetime
import requests
import pandas as pd
import sys
import time

from typing import Optional, List
from tqdm.notebook import tqdm

In [2]:
sys.version

'3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]'

In [3]:
print(f"Today is {datetime.datetime.today().strftime('%d-%b-%Y %H:%M:%S')}")

Today is 02-Feb-2026 10:18:36


In [4]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

## 1. Helper

In [5]:
def fetch_all_azure_prices(filter_query: Optional[str] = None,
                           max_pages: Optional[int] = None,
                           show_progress: bool = True,
                           retry_count: int = 3,
                           retry_delay: float = 1.0) -> pd.DataFrame:
    """
    Fetch all Azure retail prices from the API.
    
    Args:
        filter_query: Optional OData filter query
        max_pages: Optional limit on number of pages to fetch
        show_progress: Show progress bar during fetching
        retry_count: Number of retries on failure
        retry_delay: Delay between retries in seconds
    
    Returns:
        DataFrame with all Azure prices
    """
    base_url = "https://prices.azure.com/api/retail/prices"

    if filter_query:
        url = f"{base_url}?$filter={filter_query}"
    else:
        url = base_url

    all_items = []
    page_count = 0

    pbar = tqdm(desc="✅ Fetching Azure prices",
                unit=" pages") if show_progress else None

    while url:
        success = False
        for attempt in range(retry_count):
            try:
                response = requests.get(url, timeout=30)
                response.raise_for_status()
                data = response.json()
                success = True
                break
            except requests.exceptions.RequestException as e:
                if attempt < retry_count - 1:
                    print(
                        f"🔄 Retry {attempt + 1}/{retry_count} after error: {e}"
                    )
                    time.sleep(retry_delay)
                else:
                    print(
                        f"❌ Error fetching data after {retry_count} attempts: {e}"
                    )
                    break

        if not success:
            break

        items = data.get("Items", [])
        all_items.extend(items)

        page_count += 1

        if pbar is not None:
            pbar.update(1)
            pbar.set_postfix({"items": len(all_items)})

        if max_pages and page_count >= max_pages:
            print(f"Reached max pages limit ({max_pages})")
            break

        url = data.get("NextPageLink")
        time.sleep(0.1)  # Avoid rate limiting

    if pbar is not None:
        pbar.close()

    print(f"Total items fetched = {len(all_items)}")
    print(f"Total pages = {page_count}")

    return pd.DataFrame(all_items) if all_items else pd.DataFrame()

## 2. Getting prices

In [6]:
df_allprices = fetch_all_azure_prices()

✅ Fetching Azure prices: 0 pages [00:00, ? pages/s]

Total items fetched = 915302
Total pages = 916


In [7]:
len(df_allprices)

915302

In [8]:
df_allprices.head()

,currencyCode,tierMinimumUnits,retailPrice,unitPrice,armRegionName,location,effectiveStartDate,meterId,meterName,productId,skuId,productName,skuName,serviceName,serviceId,serviceFamily,unitOfMeasure,type,isPrimaryMeterRegion,armSkuName,reservationTerm,effectiveEndDate
0,USD,0.0,0.497297,0.497297,southindia,IN South,2025-10-01T00:00:00Z,000009d0-057f-5f2b-b7e9-9e26add324a8,D14/DS14 Spot,DZH318Z0BPVW,DZH318Z0BPVW/00QZ,Virtual Machines D Series Windows,D14 Spot,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_D14,NaN,NaN
1,USD,0.0,0.361284,0.361284,southindia,IN South,2025-10-01T00:00:00Z,000009d0-057f-5f2b-b7e9-9e26add324a8,D14/DS14 Spot,DZH318Z0BPVW,DZH318Z0BPVW/00QZ,Virtual Machines D Series Windows,D14 Spot,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,DevTestConsumption,True,Standard_D14,NaN,NaN
2,USD,0.0,0.101600,0.101600,uksouth,UK South,2025-06-01T00:00:00Z,ace03b73-4864-4a8c-afcb-55ddf91e010e,vCore,DZH318Z0BQJ7,DZH318Z0BQJ7/000C,Azure Database for MySQL Single Server General...,vCore,Azure Database for MySQL,DZH317NR0P99,Databases,1 Hour,Consumption,False,AzureDB_MySQL_General_Purpose_Compute_Gen5,NaN,NaN
3,USD,0.0,569.000000,569.000000,uksouth,UK South,2025-06-01T00:00:00Z,ace03b73-4864-4a8c-afcb-55ddf91e010e,vCore,DZH318Z0BQJ7,DZH318Z0BQJ7/017G,Azure Database for MySQL Single Server General...,vCore,Azure Database for MySQL,DZH317NR0P99,Databases,1 Hour,Reservation,False,AzureDB_MySQL_General_Purpose_Compute_Gen5,1 Year,NaN
4,USD,0.0,0.291000,0.291000,brazilsouth,BR South,2025-11-01T00:00:00Z,00003b45-e996-5b04-b673-a2db710f9237,D4as v7,DZH318Z0TBJJ,DZH318Z0TBJJ/00NL,Virtual Machines Dasv7 Series,D4as v7,Virtual Machines,DZH313Z7MMC8,Compute,1 Hour,Consumption,True,Standard_D4as_v7,NaN,NaN


In [9]:
df_allprices.shape

(915302, 22)

In [10]:
df_allprices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 915302 entries, 0 to 915301
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   currencyCode          915302 non-null  object 
 1   tierMinimumUnits      915302 non-null  float64
 2   retailPrice           915302 non-null  float64
 3   unitPrice             915302 non-null  float64
 4   armRegionName         915302 non-null  object 
 5   location              915302 non-null  object 
 6   effectiveStartDate    915302 non-null  object 
 7   meterId               915302 non-null  object 
 8   meterName             915302 non-null  object 
 9   productId             915302 non-null  object 
 10  skuId                 915302 non-null  object 
 11  productName           915302 non-null  object 
 12  skuName               915302 non-null  object 
 13  serviceName           915302 non-null  object 
 14  serviceId             915302 non-null  object 
 15  

In [11]:
df_allprices.columns

Index(['currencyCode', 'tierMinimumUnits', 'retailPrice', 'unitPrice',
       'armRegionName', 'location', 'effectiveStartDate', 'meterId',
       'meterName', 'productId', 'skuId', 'productName', 'skuName',
       'serviceName', 'serviceId', 'serviceFamily', 'unitOfMeasure', 'type',
       'isPrimaryMeterRegion', 'armSkuName', 'reservationTerm',
       'effectiveEndDate'],
      dtype='object')

In [12]:
df_allprices.isna().sum()

currencyCode                 0
tierMinimumUnits             0
retailPrice                  0
unitPrice                    0
armRegionName                0
location                     0
effectiveStartDate           0
meterId                      0
meterName                    0
productId                    0
skuId                        0
productName                  0
skuName                      0
serviceName                  0
serviceId                    0
serviceFamily                0
unitOfMeasure                0
type                         0
isPrimaryMeterRegion         0
armSkuName                   0
reservationTerm         776364
effectiveEndDate        907771
dtype: int64

### 2.1 Service Family

In [13]:
sf_df = df_allprices['serviceFamily'].value_counts()

pct = (sf_df / sf_df.sum() * 100).round(2)
sf_df = pd.DataFrame({
    'Count': sf_df,
    'Percentage': pct.astype(str) + '%'
}).style.background_gradient(cmap='Blues', subset=['Count'])

sf_df

,Count,Percentage
serviceFamily,,
Compute,644559,70.42%
Storage,80515,8.8%
Databases,66195,7.23%
AI + Machine Learning,35739,3.9%
Analytics,24996,2.73%
Management and Governance,13804,1.51%
Networking,13462,1.47%
Azure Communication Services,8397,0.92%
Internet of Things,5583,0.61%


### 2.2 Service Name

In [14]:
sn_df = df_allprices['serviceName'].value_counts()

pct = (sn_df / sn_df.sum() * 100).round(2)
sn_df = pd.DataFrame({
    'Count': sn_df,
    'Percentage': pct.astype(str) + '%'
}).style.background_gradient(cmap='Blues', subset=['Count'])

sn_df

,Count,Percentage
serviceName,,
Virtual Machines,610187,66.67%
Storage,75310,8.23%
SQL Database,19998,2.18%
Foundry Models,17654,1.93%
Foundry Tools,17028,1.86%
Azure App Service,14529,1.59%
Cloud Services,11576,1.26%
Azure Monitor,11155,1.22%
HDInsight,9695,1.06%


### 2.3 Product Name

In [15]:
pn_df = df_allprices['productName'].value_counts()

pct = (pn_df / pn_df.sum() * 100).round(2)
pn_df = pd.DataFrame({
    'Count': pn_df,
    'Percentage': pct.astype(str) + '%'
}).style.background_gradient(cmap='Blues', subset=['Count'])

pn_df

,Count,Percentage
productName,,
Azure Monitor,11153,1.22%
Virtual Machines Eadsv5 Series Windows,8114,0.89%
Virtual Machines Easv5 Series Windows,8114,0.89%
Virtual Machines Edsv5 Series Windows,7822,0.85%
Virtual Machines MS Series Windows,7804,0.85%
Virtual Machines Esv5 Series Windows,7640,0.83%
Azure Data Lake Storage Gen2 Hierarchical Namespace,7558,0.83%
General Block Blob v2,7556,0.83%
Virtual Machines Easv4 Series Windows,7221,0.79%


## 3. Example: Fetch Azure Content Understanding prices

In [16]:
filter_query = "contains(productName, 'Content Understanding')"

cu_df = fetch_all_azure_prices(filter_query=filter_query)

✅ Fetching Azure prices: 0 pages [00:00, ? pages/s]

Total items fetched = 151
Total pages = 1


In [17]:
print(f"Total prices = {len(cu_df)}")

Total prices = 151


In [18]:
cu_df.head()

,currencyCode,tierMinimumUnits,retailPrice,unitPrice,armRegionName,location,effectiveStartDate,meterId,meterName,productId,skuId,productName,skuName,serviceName,serviceId,serviceFamily,unitOfMeasure,type,isPrimaryMeterRegion,armSkuName
0,USD,0.0,0.00532,0.00532,swedencentral,SE Central,2025-06-01T00:00:00Z,00b6b36d-91c5-5fdd-beb6-aa3f028fc0e8,Pro Field Extract Outp Tokens,DZH318Z0ML96,DZH318Z0ML96/004F,Azure Content Understanding,Pro Field Extract Outp,Foundry Tools,DZH319RLHTZC,AI + Machine Learning,1K,Consumption,True,Pro Field Extract Outp
1,USD,0.0,1.00000,1.00000,australiaeast,AU East,2025-07-01T00:00:00Z,04bd322f-164d-5ed4-a769-fb9a51ab71f6,Face Transaction Transactions,DZH318Z0ML96,DZH318Z0ML96/004X,Azure Content Understanding,Face Transaction,Foundry Tools,DZH319RLHTZC,AI + Machine Learning,1K,Consumption,True,Face Transaction
2,USD,0.0,0.00378,0.00378,swedencentral,SE Central,2025-06-01T00:00:00Z,04d9911e-40b7-5fc7-90ae-e8b5f2e7e805,Classification Input Tokens,DZH318Z0ML96,DZH318Z0ML96/004H,Azure Content Understanding,Classification Input,Foundry Tools,DZH319RLHTZC,AI + Machine Learning,1K,Consumption,True,Classification Input
3,USD,0.0,0.80000,0.80000,swedencentral,SE Central,2024-12-01T00:00:00Z,09642389-9e29-56c5-b5e8-51e36e431ffc,Audio Field Extraction,DZH318Z0ML96,DZH318Z0ML96/001H,Azure Content Understanding,Audio Field Extraction,Foundry Tools,DZH319RLHTZC,AI + Machine Learning,1 Hour,Consumption,True,Audio Field Extraction
4,USD,0.0,0.01000,0.01000,westus3,US West 3,2026-01-01T00:00:00Z,0b39f850-0379-5f0f-b4a6-c10328bf400b,Min. Doc Content Extraction Pages,DZH318Z0ML96,DZH318Z0ML96/00FR,Azure Content Understanding,Min. Doc Content Extraction,Foundry Tools,DZH319RLHTZC,AI + Machine Learning,1K,Consumption,True,Min. Doc Content Extraction


## 4. Fetch Azure OpenAI and Microsoft Foundry Models Prices

In [19]:
# All OpenAI models
filter_query = "contains(productName, 'OpenAI')"

openai_df = fetch_all_azure_prices(filter_query=filter_query)

✅ Fetching Azure prices: 0 pages [00:00, ? pages/s]

Total items fetched = 12737
Total pages = 13


In [20]:
# All Microsoft Foundry models
filter_query = "contains(productName, 'Foundry')"

foundry_df = fetch_all_azure_prices(filter_query=filter_query)

✅ Fetching Azure prices: 0 pages [00:00, ? pages/s]

Total items fetched = 168
Total pages = 1


In [21]:
genai_df = pd.concat([openai_df, foundry_df], ignore_index=True)

In [22]:
print(f"Total prices = {len(genai_df)}")

Total prices = 12905


In [23]:
genai_df.head()

,currencyCode,tierMinimumUnits,retailPrice,unitPrice,armRegionName,location,effectiveStartDate,meterId,meterName,productId,skuId,productName,skuName,serviceName,serviceId,serviceFamily,unitOfMeasure,type,isPrimaryMeterRegion,armSkuName,reservationTerm
0,USD,0.0,35.2000,35.2000,westus3,US West 3,2025-12-01T00:00:00Z,00065cdf-d175-5834-91d3-ca200fd25770,gpt img 1.5 out img DZ 1M Tokens,DZH318Z0V4PL,DZH318Z0V4PL/0DZP,Azure OpenAI Media,gpt img 1.5 out img DZ,Foundry Models,DZH31600VF1F,AI + Machine Learning,1M,Consumption,True,gpt img 1.5 out img DZ,NaN
1,USD,0.0,0.0022,0.0022,northcentralus,US North Central,2025-05-01T00:00:00Z,0018af01-62d9-59d5-ac79-dfc7da9cf0cf,gpt-4.1-ft mdl grdr inpt Tokens,DZH318Z0HHG7,DZH318Z0HHG7/1Z76,Azure OpenAI,gpt-4.1-ft mdl grdr inpt,Foundry Models,DZH31600VF1F,AI + Machine Learning,1K,Consumption,True,gpt-4.1-ft mdl grdr inpt,NaN
2,USD,0.0,10.0000,10.0000,koreacentral,KR Central,2025-11-01T00:00:00Z,0023b56c-b47b-5ba6-bdd0-c13f3a63d774,5.1 codex opt Gl 1M Tokens,DZH318Z0T9WD,DZH318Z0T9WD/076J,Azure OpenAI GPT5,5.1 codex opt Gl,Foundry Models,DZH31600VF1F,AI + Machine Learning,1M,Consumption,True,51 codex opt Gl,NaN
3,USD,0.0,10.0000,10.0000,spaincentral,ES Central,2025-08-01T00:00:00Z,0029b2bf-6607-56b5-908a-2c77b338d3ea,GPT 5 outpt Glbl 1M Tokens,DZH318Z0T9WD,DZH318Z0T9WD/00BT,Azure OpenAI GPT5,GPT 5 outpt Glbl,Foundry Models,DZH31600VF1F,AI + Machine Learning,1M,Consumption,True,GPT 5 outpt Glbl,NaN
4,USD,0.0,0.0022,0.0022,eastus,US East,2025-11-01T00:00:00Z,002b52a5-a080-55d0-a097-60416adfb4d4,gpt 4.1 Inp regnl Tokens,DZH318Z0HHG7,DZH318Z0HHG7/1J0Q,Azure OpenAI,gpt 4.1 Inp regnl,Foundry Models,DZH31600VF1F,AI + Machine Learning,1K,Consumption,True,gpt 4.1 Inp regnl,NaN


In [24]:
pn_df = genai_df['productName'].value_counts()

pct = (pn_df / pn_df.sum() * 100).round(2)

pn_df = pd.DataFrame({
    'Count': pn_df,
    'Percentage': pct.astype(str) + '%'
}).style.background_gradient(cmap='Blues', subset=['Count'])

pn_df

,Count,Percentage
productName,,
Azure OpenAI,7122,55.19%
Azure OpenAI GPT5,2238,17.34%
Azure OpenAI Media,2162,16.75%
Azure OpenAI Reasoning,716,5.55%
Azure OpenAI OSS Models,250,1.94%
Azure OpenAI PP GPT4s,222,1.72%
Azure AI Foundry Provisioned Throughput Reservation,120,0.93%
Foundry Agents,48,0.37%
Azure OpenAI Embedding,26,0.2%


In [25]:
re_df = genai_df['armRegionName'].value_counts()

pct = (re_df / re_df.sum() * 100).round(2)

re_df = pd.DataFrame({
    'Count': re_df,
    'Percentage': pct.astype(str) + '%'
}).style.background_gradient(cmap='Blues', subset=['Count'])

re_df

,Count,Percentage
armRegionName,,
swedencentral,827,6.41%
eastus2,804,6.23%
northcentralus,788,6.11%
eastus,745,5.77%
westus,728,5.64%
southcentralus,724,5.61%
westus3,719,5.57%
francecentral,631,4.89%
westeurope,625,4.84%


In [26]:
sku_df = genai_df['skuName'].value_counts()

pct = (sku_df / sku_df.sum() * 100).round(2)

sku_df = pd.DataFrame({
    'Count': sku_df,
    'Percentage': pct.astype(str) + '%'
}).style.background_gradient(cmap='Blues', subset=['Count'])

sku_df

,Count,Percentage
skuName,,
Provisioned Managed Regional,75,0.58%
OSS-20b FT,72,0.56%
Provisioned Managed Global,72,0.56%
Hosted HOBO,48,0.37%
Provisioned Managed Data Zone,39,0.3%
gpt-oss-120B Inp glbl,36,0.28%
GPT-OSS-20b-IN-FT,36,0.28%
gpt-oss-120B Outp glbl,36,0.28%
GPT-OSS-20b-Out-FT,36,0.28%


In [27]:
sku_list = genai_df['skuName'].value_counts().index.tolist()
sku_list = sorted(sku_list)
sku_list

['5.1 codex cd inp Dz',
 '5.1 codex cd inp Gl',
 '5.1 codex inp Dz',
 '5.1 codex inp Gl',
 '5.1 codex max cd inp Dz',
 '5.1 codex max cd inp Gl',
 '5.1 codex max inp Dz',
 '5.1 codex max inp Gl',
 '5.1 codex max opt Dz',
 '5.1 codex max opt Gl',
 '5.1 codex mini cd inp Dz',
 '5.1 codex mini cd inp Gl',
 '5.1 codex mini inp Dz',
 '5.1 codex mini inp Gl',
 '5.1 codex mini opt Dz',
 '5.1 codex mini opt Gl',
 '5.1 codex opt Dz',
 '5.1 codex opt Gl',
 '5.2 codex cd inp Dz',
 '5.2 codex cd inp Gl',
 '5.2 codex inp Dz',
 '5.2 codex inp Gl',
 '5.2 codex opt Dz',
 '5.2 codex opt Gl',
 'Assistants-File Search-glbl',
 'Assistants-File Search-regnl',
 'Az-Babbage-002',
 'Az-Babbage-002-Fine Tuned-Input',
 'Az-Babbage-002-Fine Tuned-Output',
 'Az-Davinci-002',
 'Az-Davinci-002-Fine Tuned-Input',
 'Az-Davinci-002-Fine Tuned-Output',
 'Az-Embeddings-Ada',
 'Az-GPT-3.5-turbo',
 'Az-GPT35-Turbo-16K-1106 Input',
 'Az-GPT35-Turbo-16K-1106 Output',
 'Az-GPT4-Turbo-128K Input',
 'Az-GPT4-Turbo-128K Output'

### Pricing for gpt 5.1

In [28]:
result = list(filter(lambda item: 'GPT 5.1' in item, sku_list))
result

['GPT 5.1 Batch cd inp Dz',
 'GPT 5.1 Batch cd inp Gl',
 'GPT 5.1 Batch inp Dz',
 'GPT 5.1 Batch inp Gl',
 'GPT 5.1 Batch opt Dz',
 'GPT 5.1 Batch opt Gl',
 'GPT 5.1 cd inp Dz',
 'GPT 5.1 cd inp Gl',
 'GPT 5.1 chat cd inp Dz',
 'GPT 5.1 chat cd inp Gl',
 'GPT 5.1 chat inp Dz',
 'GPT 5.1 chat inp Gl',
 'GPT 5.1 chat opt Dz',
 'GPT 5.1 chat opt Gl',
 'GPT 5.1 inp Dz',
 'GPT 5.1 inp Gl',
 'GPT 5.1 opt Dz',
 'GPT 5.1 opt Gl']

In [29]:
model_input = "GPT 5.1 inp Gl"  # inp is for input. Gl is for GLOBAL
model_output = "GPT 5.1 opt Gl"  # opt is for output. Gl is for GLOBAL

In [30]:
result_input = genai_df[genai_df['skuName'] == model_input]
result_output = genai_df[genai_df['skuName'] == model_output]
results = pd.concat([result_input, result_output], ignore_index=True)

In [31]:
results.head()

,currencyCode,tierMinimumUnits,retailPrice,unitPrice,armRegionName,location,effectiveStartDate,meterId,meterName,productId,skuId,productName,skuName,serviceName,serviceId,serviceFamily,unitOfMeasure,type,isPrimaryMeterRegion,armSkuName,reservationTerm
0,USD,0.0,1.25,1.25,canadaeast,CA East,2025-11-01T00:00:00Z,28319d20-7d1d-5dd4-90fe-70cecabb2d80,GPT 5.1 inp Gl 1M Tokens,DZH318Z0T9WD,DZH318Z0T9WD/059W,Azure OpenAI GPT5,GPT 5.1 inp Gl,Foundry Models,DZH31600VF1F,AI + Machine Learning,1M,Consumption,True,GPT 51 inp Gl,NaN
1,USD,0.0,1.25,1.25,eastus,US East,2025-11-01T00:00:00Z,2d126542-7648-5bd4-bff3-070731cc772d,GPT 5.1 inp Gl 1M Tokens,DZH318Z0T9WD,DZH318Z0T9WD/06HB,Azure OpenAI GPT5,GPT 5.1 inp Gl,Foundry Models,DZH31600VF1F,AI + Machine Learning,1M,Consumption,True,GPT 51 inp Gl,NaN
2,USD,0.0,1.25,1.25,brazilsouth,BR South,2025-11-01T00:00:00Z,2ea4209f-6102-5c9e-b84f-d1bf822cde64,GPT 5.1 inp Gl 1M Tokens,DZH318Z0T9WD,DZH318Z0T9WD/05CZ,Azure OpenAI GPT5,GPT 5.1 inp Gl,Foundry Models,DZH31600VF1F,AI + Machine Learning,1M,Consumption,True,GPT 51 inp Gl,NaN
3,USD,0.0,1.25,1.25,southafricanorth,ZA North,2025-11-01T00:00:00Z,5325c68c-ae58-5127-9871-3d61050e3472,GPT 5.1 inp Gl 1M Tokens,DZH318Z0T9WD,DZH318Z0T9WD/06NZ,Azure OpenAI GPT5,GPT 5.1 inp Gl,Foundry Models,DZH31600VF1F,AI + Machine Learning,1M,Consumption,True,GPT 51 inp Gl,NaN
4,USD,0.0,1.25,1.25,swedencentral,SE Central,2025-11-01T00:00:00Z,595fe59b-03a8-5806-83b0-a49e59be6435,GPT 5.1 inp Gl 1M Tokens,DZH318Z0T9WD,DZH318Z0T9WD/05S2,Azure OpenAI GPT5,GPT 5.1 inp Gl,Foundry Models,DZH31600VF1F,AI + Machine Learning,1M,Consumption,True,GPT 51 inp Gl,NaN


In [32]:
region = "swedencentral"

In [33]:
region_results = results[results['armRegionName'] == region]
region_results

,currencyCode,tierMinimumUnits,retailPrice,unitPrice,armRegionName,location,effectiveStartDate,meterId,meterName,productId,skuId,productName,skuName,serviceName,serviceId,serviceFamily,unitOfMeasure,type,isPrimaryMeterRegion,armSkuName,reservationTerm
4,USD,0.0,1.25,1.25,swedencentral,SE Central,2025-11-01T00:00:00Z,595fe59b-03a8-5806-83b0-a49e59be6435,GPT 5.1 inp Gl 1M Tokens,DZH318Z0T9WD,DZH318Z0T9WD/05S2,Azure OpenAI GPT5,GPT 5.1 inp Gl,Foundry Models,DZH31600VF1F,AI + Machine Learning,1M,Consumption,True,GPT 51 inp Gl,NaN
36,USD,0.0,10.00,10.00,swedencentral,SE Central,2025-11-01T00:00:00Z,8a220f8d-9d2e-5a3b-a119-ef0ad5a89092,GPT 5.1 opt Gl 1M Tokens,DZH318Z0T9WD,DZH318Z0T9WD/05QP,Azure OpenAI GPT5,GPT 5.1 opt Gl,Foundry Models,DZH31600VF1F,AI + Machine Learning,1M,Consumption,True,GPT 51 opt Gl,NaN


In [34]:
print("\033[1;31;34m")
print(f"{'Model: GPT 5.1 Global':^50}")
print(f"{'Region: ' + region:^50}")
print(f"{'-'*50}")
print(f"Input tokens:  ${region_results.iloc[0]['retailPrice']}")
print(f"Output tokens: ${region_results.iloc[1]['retailPrice']}")
print(f"Unit:          {region_results.iloc[0]['unitOfMeasure']}")


              Model: GPT 5.1 Global               
              Region: swedencentral               
--------------------------------------------------
Input tokens:  $1.25
Output tokens: $10.0
Unit:          1M


In [35]:
print("\033[1;31;34m")
for idx, row in region_results.iterrows():
    for col, value in row.items():
        print(f"{col:25}: {value}")
    print()


currencyCode             : USD
tierMinimumUnits         : 0.0
retailPrice              : 1.25
unitPrice                : 1.25
armRegionName            : swedencentral
location                 : SE Central
effectiveStartDate       : 2025-11-01T00:00:00Z
meterId                  : 595fe59b-03a8-5806-83b0-a49e59be6435
meterName                : GPT 5.1 inp Gl 1M Tokens
productId                : DZH318Z0T9WD
skuId                    : DZH318Z0T9WD/05S2
productName              : Azure OpenAI GPT5
skuName                  : GPT 5.1 inp Gl
serviceName              : Foundry Models
serviceId                : DZH31600VF1F
serviceFamily            : AI + Machine Learning
unitOfMeasure            : 1M
type                     : Consumption
isPrimaryMeterRegion     : True
armSkuName               : GPT 51 inp Gl
reservationTerm          : nan

currencyCode             : USD
tierMinimumUnits         : 0.0
retailPrice              : 10.0
unitPrice                : 10.0
armRegionName            : 

### Another model pricing

In [36]:
model_input = "gpt 4.1 mini Outp glbl"
model_output = "gpt 4.1 mini Inp glbl"

In [37]:
result_input = genai_df[genai_df['skuName'] == model_input]
result_output = genai_df[genai_df['skuName'] == model_output]
results = pd.concat([result_input, result_output], ignore_index=True)
results.head()

,currencyCode,tierMinimumUnits,retailPrice,unitPrice,armRegionName,location,effectiveStartDate,meterId,meterName,productId,skuId,productName,skuName,serviceName,serviceId,serviceFamily,unitOfMeasure,type,isPrimaryMeterRegion,armSkuName,reservationTerm
0,USD,0.0,0.0016,0.0016,centralus,US Central,2025-11-01T00:00:00Z,0aa0219b-d2f5-5146-a0b8-7a98518a0a17,gpt 4.1 mini Outp glbl Tokens,DZH318Z0HHG7,DZH318Z0HHG7/1L36,Azure OpenAI,gpt 4.1 mini Outp glbl,Foundry Models,DZH31600VF1F,AI + Machine Learning,1K,Consumption,True,gpt 4.1 mini Outp glbl,NaN
1,USD,0.0,0.0016,0.0016,switzerlandnorth,CH North,2025-04-01T00:00:00Z,0e988adb-396a-516d-8dfc-b48bde774a35,gpt 4.1 mini Outp glbl Tokens,DZH318Z0HHG7,DZH318Z0HHG7/1Q4D,Azure OpenAI,gpt 4.1 mini Outp glbl,Foundry Models,DZH31600VF1F,AI + Machine Learning,1K,Consumption,True,gpt 4.1 mini Outp glbl,NaN
2,USD,0.0,0.0016,0.0016,southafricanorth,ZA North,2025-04-01T00:00:00Z,0f346c58-dd6e-563d-af13-4832dc7bf844,gpt 4.1 mini Outp glbl Tokens,DZH318Z0HHG7,DZH318Z0HHG7/1PJW,Azure OpenAI,gpt 4.1 mini Outp glbl,Foundry Models,DZH31600VF1F,AI + Machine Learning,1K,Consumption,True,gpt 4.1 mini Outp glbl,NaN
3,USD,0.0,0.0016,0.0016,francecentral,FR Central,2025-04-01T00:00:00Z,0fe19da4-bf69-532f-8c8c-a8bcd501bb61,gpt 4.1 mini Outp glbl Tokens,DZH318Z0HHG7,DZH318Z0HHG7/1PNH,Azure OpenAI,gpt 4.1 mini Outp glbl,Foundry Models,DZH31600VF1F,AI + Machine Learning,1K,Consumption,True,gpt 4.1 mini Outp glbl,NaN
4,USD,0.0,0.0016,0.0016,polandcentral,PL Central,2025-04-01T00:00:00Z,0fe80feb-64c8-5477-8f96-23e7d1e04241,gpt 4.1 mini Outp glbl Tokens,DZH318Z0HHG7,DZH318Z0HHG7/1MTR,Azure OpenAI,gpt 4.1 mini Outp glbl,Foundry Models,DZH31600VF1F,AI + Machine Learning,1K,Consumption,True,gpt 4.1 mini Outp glbl,NaN


In [38]:
region_results = results[results['armRegionName'] == region]
region_results

,currencyCode,tierMinimumUnits,retailPrice,unitPrice,armRegionName,location,effectiveStartDate,meterId,meterName,productId,skuId,productName,skuName,serviceName,serviceId,serviceFamily,unitOfMeasure,type,isPrimaryMeterRegion,armSkuName,reservationTerm
20,USD,0.0,0.0016,0.0016,swedencentral,SE Central,2025-04-01T00:00:00Z,b988c77d-78c1-5d70-8dfa-4e841f490ff8,gpt 4.1 mini Outp glbl Tokens,DZH318Z0HHG7,DZH318Z0HHG7/1P75,Azure OpenAI,gpt 4.1 mini Outp glbl,Foundry Models,DZH31600VF1F,AI + Machine Learning,1K,Consumption,True,gpt 4.1 mini Outp glbl,NaN
39,USD,0.0,0.0004,0.0004,swedencentral,SE Central,2025-04-01T00:00:00Z,6ee3bd5d-85f1-543b-b478-ec44c7825072,gpt 4.1 mini Inp glbl Tokens,DZH318Z0HHG7,DZH318Z0HHG7/1P6G,Azure OpenAI,gpt 4.1 mini Inp glbl,Foundry Models,DZH31600VF1F,AI + Machine Learning,1K,Consumption,True,gpt 4.1 mini Inp glbl,NaN


In [39]:
print("\033[1;31;34m")
print(f"{'Model: GPT 4.1 mini Global':^50}")
print(f"{'Region: ' + region:^50}")
print(f"{'-'*50}")
print(f"Input tokens:  ${region_results.iloc[1]['retailPrice']}")
print(f"Output tokens: ${region_results.iloc[0]['retailPrice']}")
print(f"Unit:          {region_results.iloc[0]['unitOfMeasure']}")


            Model: GPT 4.1 mini Global            
              Region: swedencentral               
--------------------------------------------------
Input tokens:  $0.0004
Output tokens: $0.0016
Unit:          1K


In [40]:
print("\033[1;31;34m")
for idx, row in region_results.iterrows():
    for col, value in row.items():
        print(f"{col:25}: {value}")
    print()


currencyCode             : USD
tierMinimumUnits         : 0.0
retailPrice              : 0.0016
unitPrice                : 0.0016
armRegionName            : swedencentral
location                 : SE Central
effectiveStartDate       : 2025-04-01T00:00:00Z
meterId                  : b988c77d-78c1-5d70-8dfa-4e841f490ff8
meterName                : gpt 4.1 mini Outp glbl Tokens
productId                : DZH318Z0HHG7
skuId                    : DZH318Z0HHG7/1P75
productName              : Azure OpenAI
skuName                  : gpt 4.1 mini Outp glbl
serviceName              : Foundry Models
serviceId                : DZH31600VF1F
serviceFamily            : AI + Machine Learning
unitOfMeasure            : 1K
type                     : Consumption
isPrimaryMeterRegion     : True
armSkuName               : gpt 4.1 mini Outp glbl
reservationTerm          : nan

currencyCode             : USD
tierMinimumUnits         : 0.0
retailPrice              : 0.0004
unitPrice                : 0.0004
ar

## 5. Saving results

In [41]:
SAVE_DIR = "results"

os.makedirs(SAVE_DIR, exist_ok=True)

In [42]:
now = datetime.datetime.today().strftime('%d%b%Y_%H%M%S').upper()

In [43]:
csv_file = os.path.join(SAVE_DIR, f"azure_all_prices_{now}.csv")

df_allprices.to_csv(csv_file, index=False)
!ls $csv_file -lh

-rwxrwxrwx 1 root root 245M Feb  2 10:25 results/azure_all_prices_02FEB2026_102539.csv


In [44]:
csv_file2 = os.path.join(SAVE_DIR, f"azure_genai_prices_{now}.csv")

genai_df.to_csv(csv_file2, index=False)
!ls $csv_file2 -lh

-rwxrwxrwx 1 root root 3.8M Feb  2 10:25 results/azure_genai_prices_02FEB2026_102539.csv


In [46]:
excel_file = os.path.join(SAVE_DIR, f"azure_genai_prices_{now}.xlsx")

genai_df.to_excel(excel_file, index=False)
!ls $excel_file -lh

-rwxrwxrwx 1 root root 1.7M Feb  2 10:26 results/azure_genai_prices_02FEB2026_102539.xlsx
